# Week 2 · Module 3 Practical — Real Infrastructure 🗄️

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 2 · Module 3: Building a vector store with ChromaDB — indexing, searching, HNSW & hybrid retrieval**

---

### What you'll do in the next 30 minutes

| # | Experiment | Skill |
|---|-----------|-------|
| 1 | Index into ChromaDB | collections, ids, documents, embeddings, metadata |
| 2 | Filter + search | `where` clauses: "kitchen under ₹1,000" |
| 3 | The timing race | numpy brute force vs **HNSW** at 50,000 vectors |
| 4 | Hybrid via **RRF** | merge BM25 + vector rankings in ~10 lines |
| 5 | 3-way shootout | BM25 vs vectors vs hybrid on the gold set |
| 🏁 | **RAGBot v0.3** | database-backed, hybrid, filter-capable |

> ⏳ The `chromadb` install takes ~1 minute — run Step 0 now and read ahead while it works.

## Step 0 — Setup 🔑

In [ ]:
# chromadb is the big one — start this first (leave Colab's numpy alone!)
%pip install -q -U openai rank_bm25 chromadb

In [ ]:
from getpass import getpass
from openai import OpenAI
import numpy as np
import chromadb

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def ask(prompt, temperature=0.3, max_tokens=250):
    r = client.chat.completions.create(model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

def embed_batch(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

print("✅ Ready:", CHAT_MODEL, "+", EMBED_MODEL, "+ chromadb", chromadb.__version__)

In [ ]:
# The knowledge base — now with STRUCTURED fields for metadata
# (id, text, category, price_in_rupees or None for policies)
KB = [
    ("P-101", "Prima Electric Kettle 1.5L | Rs. 1,299 | 1500W fast boil, auto shut-off, 1-year warranty | in stock: 42", "kitchen", 1299),
    ("P-102", "Prima Electric Kettle 2.0L | Rs. 1,699 | family size, keep-warm mode, 1-year warranty | in stock: 18", "kitchen", 1699),
    ("P-103", "Zenith Steel Kettle 1.0L | Rs. 899 | budget stovetop kettle, no electricity needed | in stock: 65", "kitchen", 899),
    ("P-104", "Zenith Pro Kettle 1.8L | Rs. 2,499 | premium, temperature control, 2-year warranty | in stock: 7", "kitchen", 2499),
    ("P-105", "SwiftMix Mixer Grinder 750W | Rs. 3,499 | 3 jars, overload protection, 2-year warranty | in stock: 23", "kitchen", 3499),
    ("P-106", "SwiftMix Mixer Grinder 500W | Rs. 2,299 | 2 jars, compact, 1-year warranty | in stock: 31", "kitchen", 2299),
    ("P-107", "AirChef Air Fryer 4L | Rs. 5,999 | oil-free frying, digital timer, 1-year warranty | in stock: 12", "kitchen", 5999),
    ("P-108", "AirChef Air Fryer 6L XL | Rs. 8,499 | family size, 8 presets, 2-year warranty | in stock: 4", "kitchen", 8499),
    ("P-109", "CoolBreeze Table Fan 400mm | Rs. 1,899 | 3-speed, oscillating, 2-year warranty | in stock: 55", "appliances", 1899),
    ("P-110", "CoolBreeze Tower Fan | Rs. 4,299 | remote control, timer, quiet mode | in stock: 9", "appliances", 4299),
    ("P-111", "BrightHome LED Bulb 9W (pack of 4) | Rs. 499 | warm white, 2-year replacement | in stock: 210", "electrical", 499),
    ("P-112", "BrightHome Smart Bulb WiFi | Rs. 799 | app control, 16M colors, voice assistant support | in stock: 48", "electrical", 799),
    ("P-113", "PowerSafe Extension Board 6-socket | Rs. 649 | surge protection, 2m cord | in stock: 87", "electrical", 649),
    ("P-114", "SmartWatch Fit Pro | Rs. 2,999 | heart rate, SpO2, 7-day battery | in stock: 26", "gadgets", 2999),
    ("P-115", "EarBuds Bass+ TWS | Rs. 1,499 | 24h playback, IPX4, ENC mic | in stock: 39", "gadgets", 1499),
    ("P-116", "PixelSnap Barcode Scanner X-500 | Rs. 4,999 | wireless, 18h battery, 1-year warranty | in stock: 6", "business", 4999),
    ("P-117", "ThermoSteel Water Bottle 1L | Rs. 749 | 12h hot / 24h cold, leak-proof | in stock: 120", "home", 749),
    ("P-118", "CleanSweep Robot Vacuum | Rs. 12,999 | app control, auto-dock, HEPA filter | in stock: 3", "home", 12999),
    ("P-119", "FreshBrew Coffee Maker 600ml | Rs. 2,799 | drip brew, keep-warm plate, 1-year warranty | in stock: 14", "kitchen", 2799),
    ("P-120", "ChopMaster Vegetable Chopper | Rs. 599 | 900ml, 3 blades, dishwasher safe | in stock: 73", "kitchen", 599),
    ("D-201", "POLICY delivery | Free delivery above Rs. 999, otherwise Rs. 49. Standard 2-4 days in Pune, 4-7 days rest of Maharashtra.", "policy", None),
    ("D-202", "POLICY returns | Returns within 7 days with receipt for full refund. Without receipt: store credit only. Electronics must be unopened.", "policy", None),
    ("D-203", "POLICY warranty | Warranty claims need invoice + product serial number. Service center: SmartMart Pimple Saudagar, 9 AM-9 PM.", "policy", None),
    ("D-204", "POLICY offers | Current offer: 10% off all kitchen appliances till Sunday. Not combinable with other coupons.", "policy", None),
]
IDS       = [r[0] for r in KB]
DOCS      = [f"{r[0]} | {r[1]}" for r in KB]
METAS     = [{"category": r[2], "price": r[3] if r[3] is not None else -1} for r in KB]
print(f"{len(KB)} chunks with metadata ready")

---
## Experiment 1 — Index into ChromaDB 🗄️

Yesterday you managed ids, texts and vectors in parallel numpy arrays by hand. A collection stores them **together**, keyed by ID — and builds an **HNSW index** over the vectors automatically:

In [ ]:
chroma = chromadb.Client()                     # in-memory; PersistentClient(path=...) for disk

# fresh start if you re-run this cell
try: chroma.delete_collection("smartmart")
except Exception: pass

col = chroma.create_collection(
    name="smartmart",
    metadata={"hnsw:space": "cosine"},         # the distance metric WE choose
)

vecs = embed_batch(DOCS)                       # our OpenAI vectors, same as yesterday
col.add(ids=IDS, documents=DOCS, embeddings=vecs.tolist(), metadatas=METAS)

print(f"Collection '{col.name}': {col.count()} items, cosine space, HNSW index built")

In [ ]:
def retrieve_chroma(query, k=3, where=None):
    qv = embed_batch([query])[0].tolist()
    res = col.query(query_embeddings=[qv], n_results=k, where=where)
    # chroma returns DISTANCE (cosine distance = 1 - similarity); flip for familiarity
    return [(doc, 1 - dist) for doc, dist in zip(res["documents"][0], res["distances"][0])]

for doc, sim in retrieve_chroma("sasta kettle dikhao", k=3):
    print(f"[{sim:.3f}] {doc[:75]}...")

**Same vectors, same results as yesterday's numpy** — but now they're in a database: persistent-capable, updatable by ID (`col.update`, `col.delete`), and searched by HNSW instead of brute force.

**✏️ Your turn:** run `col.get(ids=["P-103"])` — a vector DB is still a database; you can look things up by key.

---
## Experiment 2 — Filter first, then search 🏷️

*"Something for making tea, under ₹1,000, kitchen section."* Similarity can't enforce `< 1000` — numbers are where vectors blur. **Metadata filters are exact:**

In [ ]:
query = "something for making tea"

print("=== No filter ===")
for doc, sim in retrieve_chroma(query, k=3):
    print(f"[{sim:.3f}] {doc[:70]}...")

print()
print("=== where: kitchen AND price < 1000 ===")
for doc, sim in retrieve_chroma(query, k=3, where={
        "$and": [{"category": {"$eq": "kitchen"}},
                 {"price":    {"$lt": 1000}}]}):
    print(f"[{sim:.3f}] {doc[:70]}...")

**The filtered run can only return kitchen items under ₹1,000** — the constraint is set-logic, applied exactly; similarity ranks whatever survives. The unfiltered run happily suggests a ₹2,499 kettle.

**✏️ Your turn:** find every product under ₹800 that isn't electrical. (Hint: `$ne`, `$lt`.) Design rule: anything a customer can constrain exactly — price, category, stock, brand — belongs in metadata at index time.

---
## Experiment 3 — The timing race: brute force vs HNSW 🏁

Our 24 chunks can't show why HNSW exists. So: **50,000 synthetic vectors**, both methods, wall-clock time.

In [ ]:
import time

N, DIM = 50_000, 256
rng = np.random.default_rng(42)
big = rng.standard_normal((N, DIM)).astype(np.float32)
big = big / np.linalg.norm(big, axis=1, keepdims=True)
queries = big[:20] + 0.05 * rng.standard_normal((20, DIM)).astype(np.float32)  # near-duplicates of known rows

# --- contender 1: numpy brute force ---
t0 = time.perf_counter()
for q in queries:
    scores = big @ (q / np.linalg.norm(q))
    top = np.argsort(scores)[::-1][:5]
brute_ms = (time.perf_counter() - t0) / len(queries) * 1000

# --- contender 2: ChromaDB's HNSW ---
try: chroma.delete_collection("race")
except Exception: pass
race = chroma.create_collection("race", metadata={"hnsw:space": "cosine"})
t0 = time.perf_counter()
B = 5000                                             # chroma prefers batched adds
for i in range(0, N, B):
    race.add(ids=[str(j) for j in range(i, min(i+B, N))],
             embeddings=big[i:i+B].tolist())
build_s = time.perf_counter() - t0

t0 = time.perf_counter()
for q in queries:
    race.query(query_embeddings=[q.tolist()], n_results=5)
hnsw_ms = (time.perf_counter() - t0) / len(queries) * 1000

print(f"Corpus: {N:,} vectors x {DIM} dims")
print(f"Brute force : {brute_ms:7.2f} ms/query   (checks all {N:,} every time)")
print(f"HNSW        : {hnsw_ms:7.2f} ms/query   (index build took {build_s:.1f}s, once)")
print(f"Speedup     : {brute_ms/hnsw_ms:5.1f}x  — and the gap GROWS with corpus size")

**Read the trade like an engineer:**
- HNSW **pays once** at index time (the build seconds) to answer **every query** in ~constant-ish time
- Brute force pays linearly on **every single query**, forever
- At 24 chunks brute force actually wins (no index overhead) — at 50K it's no contest, at 10M it's not a choice

The fine print from the slides applies: HNSW is *approximate* (recall <100%, tunable via `ef`), and it eats RAM. Engineering is choosing trade-offs on purpose.

---
## Experiment 4 — Hybrid search with RRF 🤝

Both engines, one ranking. **Reciprocal Rank Fusion**: `score(doc) = Σ 1/(60 + rank)` across rankers — ranks are comparable even when raw scores aren't:

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(t):
    return t.lower().replace("|", " ").replace(",", " ").split()

bm25 = BM25Okapi([tokenize(d) for d in DOCS])

def retrieve_bm25(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(range(len(DOCS)), key=lambda i: scores[i], reverse=True)[:k]
    return [(DOCS[i], scores[i]) for i in ranked]

def retrieve_hybrid(query, k=3, pool=10, rrf_k=60):
    """RRF-merge BM25 and vector rankings. ~10 lines, industry standard."""
    bm_docs  = [d for d, _ in retrieve_bm25(query, k=pool)]
    vec_docs = [d for d, _ in retrieve_chroma(query, k=pool)]
    scores = {}
    for ranking in (bm_docs, vec_docs):
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (rrf_k + rank)
    merged = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
    return merged

# The two nemesis queries — one from each failure family:
for q in ["X-500", "sasta kettle dikhao"]:
    print(f"Q: {q}")
    print(f"  BM25 top   : {retrieve_bm25(q, k=1)[0][0][:55]}...")
    print(f"  Vector top : {retrieve_chroma(q, k=1)[0][0][:55]}...")
    print(f"  HYBRID top : {retrieve_hybrid(q, k=1)[0][0][:55]}...")
    print()

**One retriever that survives both nemesis queries.** BM25's exact-string strength carries "X-500"; the vectors carry "sasta kettle"; RRF lets whichever engine is confident win — without ever comparing their incomparable raw scores.

**✏️ Your turn:** print the full top-3 RRF scores for "X-500 warranty claim process" — you should see the scanner AND the warranty policy both surface (one from each engine).

---
## Experiment 5 — The 3-way shootout 📊

Same gold set as Days 1–2. Three retrievers. Four metrics. One table:

In [ ]:
TEST_SET = [
    ("price of prima 1.5L kettle",              {"P-101"}),
    ("cheapest kettle",                          {"P-103"}),
    ("robot vacuum in stock?",                   {"P-118"}),
    ("X-500 scanner warranty",                   {"P-116", "D-203"}),
    ("return policy without receipt",            {"D-202"}),
    ("delivery charge for Rs. 500 order",        {"D-201"}),
    ("discount on kitchen items",                {"D-204"}),
    ("mixer grinder with overload protection",   {"P-105"}),
    ("affordable way to boil water",             {"P-103", "P-101"}),
    ("smart lighting for home",                  {"P-112"}),
    ("sasta kettle dikhao",                      {"P-103"}),
    ("something to make coffee in the morning",  {"P-119"}),
    ("gift idea: music on the go",               {"P-115"}),
]

def chunk_id(doc):
    return doc.split("|")[0].strip().split()[0]

def evaluate(retriever, k=3):
    ps, rs, f1s, rrs = [], [], [], []
    for query, relevant in TEST_SET:
        retrieved_ids = [chunk_id(d) for d, _ in retriever(query, k=k)]
        hits = [rid for rid in retrieved_ids if rid in relevant]
        p = len(hits) / k
        r = len(set(hits)) / len(relevant)
        f1s.append(2*p*r/(p+r) if (p+r) else 0.0)
        ps.append(p); rs.append(r)
        rrs.append(next((1.0/rank for rank, rid in enumerate(retrieved_ids, 1) if rid in relevant), 0.0))
    n = len(TEST_SET)
    return {"P@k": sum(ps)/n, "R@k": sum(rs)/n, "F1": sum(f1s)/n, "MRR": sum(rrs)/n}

results = {
    "BM25":   evaluate(retrieve_bm25),
    "Vector": evaluate(retrieve_chroma),
    "Hybrid": evaluate(retrieve_hybrid),
}
print(f"{'metric':<6}" + "".join(f"{name:>10}" for name in results))
for m in ["P@k", "R@k", "F1", "MRR"]:
    print(f"{m:<6}" + "".join(f"{res[m]:>10.2f}" for res in results.values()))

**Expected shape:** Vector beats BM25 overall (the synonym queries), and **Hybrid ties or beats the best single retriever** — while being the only one strong on *both* failure families. On 24 chunks the margins are small; on real mixed traffic the stability is the point.

**✏️ Your turn:** change `rrf_k` from 60 to 10 and re-run. A smaller constant lets each engine's #1 dominate more — did it help or hurt here? (You just tuned a retrieval hyperparameter against a gold set. That's Day 4 thinking.)

---
## 🏁 Finale — RAGBot v0.3

In [ ]:
RAG_TEMPLATE = """You are SmartBot, the customer assistant of SmartMart retail store, Pune.

Answer ONLY from the context below. If the answer is not in the context,
say "I don't have that information — let me connect you with our team."
Cite the product/policy ID you used, like [P-101].
Answer in max 3 sentences, warm and professional.

CONTEXT:
{context}

CUSTOMER QUESTION: {question}"""

class RAGBotV3:
    """v0.3 — ChromaDB-backed, hybrid retrieval, optional metadata filters."""

    def __init__(self, k=3):
        self.k = k

    def say(self, question, where=None, verbose=False):
        if where:                                   # filtered questions skip hybrid (BM25 has no filters)
            chunks = [d for d, _ in retrieve_chroma(question, k=self.k, where=where)]
        else:
            chunks = [d for d, _ in retrieve_hybrid(question, k=self.k)]
        if verbose:
            for c in chunks: print(f"   ⤷ {c[:75]}...")
        return ask(RAG_TEMPLATE.format(context="\n".join(chunks), question=question))

bot = RAGBotV3(k=3)
print("🛒", bot.say("what's the warranty process for the X-500?"))
print()
print("🛒", bot.say("sasta kettle chahiye, koi discount hai kya?"))
print()
print("🛒", bot.say("suggest something for tea lovers",
                    where={"$and": [{"category": {"$eq": "kitchen"}},
                                    {"price": {"$lt": 1000}}]}))

In [ ]:
# 💬 Chat with v0.3 — 'v ' prefix for verbose retrieval, 'quit' to stop.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 v0.3 shipped. Tomorrow we grade the whole pipeline with RAGAS.")
        break
    verbose = user_msg.startswith("v ")
    print("🛒", bot.say(user_msg[2:] if verbose else user_msg, verbose=verbose))

---
## 🏆 Homework (15 min, feeds Friday's graded mini-project)

1. **Make it survive a restart** — switch to `chromadb.PersistentClient(path="./smartmart_db")`, re-index, restart the runtime, and prove the collection loads *without re-embedding* (count the embedding calls!). Persistence = not paying twice.
2. **Tune the fusion** — sweep `rrf_k` over [10, 30, 60, 100] and `pool` over [5, 10, 20]; report the best F1/MRR combo on the gold set. Two hyperparameters, one gold set — this is real retrieval engineering.
3. **Extend the store** — add 3 products of your own invention *with correct metadata*, one gold-set query each, and show hybrid retrieval finds them. Note what you did NOT have to do: retrain anything.

### What's next
**Module 4 — the evaluation deep dive:** the retrieval metrics formalised, plus **RAGAS** — faithfulness, answer relevancy, context precision & recall — grading not just what you *fetched* but what the bot *said*. Your v0.3 becomes measurable end-to-end.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*